# 03 Temporal Validation And Model Sweep

Pull same-pitcher 2026 rows when available, then compare empirical lookup, logistic, random forest, and boosted models.

In [ ]:
# Colab/local setup. Keep helper functions at the top of every notebook.
import os
import sys
import subprocess
from pathlib import Path

# Check if the code is running in Google Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""
# Define a variable for the top pitchers, in this case top 100
TOP_N = 100

# Conditional setup for Colab environment
if IN_COLAB:
    # Install required Python packages
    %pip -q install -r requirements.txt || %pip -q install pybaseball pandas numpy scikit-learn matplotlib seaborn pyarrow shap statsmodels xgboost tqdm
    # If REPO_URL is provided and the directory doesn't exist, clone the GitHub repository
    if REPO_URL and not Path("/content/pitch-sequencing").exists():
        !git clone {REPO_URL} /content/pitch-sequencing
    # base diretory for proejct
    BASE_DIR = Path("/content/pitch-sequencing") if Path("/content/pitch-sequencing").exists() else Path.cwd()
else:
    BASE_DIR = Path.cwd()

os.chdir(BASE_DIR)
sys.path.insert(0, str(BASE_DIR / "code"))

# Define paths for different data and output directories
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "output"

# Create the necessary directories if they don't already exist
for path in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, OUTPUT_DIR / "figures"]:
    path.mkdir(parents=True, exist_ok=True)

# Helper function to run external command-line steps
def run_step(args):
    print("Running:", " ".join(map(str, args))) # Print the command being executed
    # Execute the command using subprocess, capturing output
    result = subprocess.run(args, cwd=BASE_DIR, text=True, capture_output=True)
    print(result.stdout) # Print standard output
    if result.stderr:
        print(result.stderr) # Print standard error if any
    result.check_returncode() # Raise an exception if the command returned a non-zero exit code

# Helper function to display basic diagnostics of a DataFrame
def frame_diag(df, label):
    print(f"{label}: rows={len(df):,}, cols={df.shape[1]:,}") # Print number of rows and columns
    if "pitcher_name" in df: # If 'pitcher_name' column exists, print unique pitcher count
        print(f"{label}: pitchers={df['pitcher_name'].nunique():,}")
    if "pitch_type" in df: # If 'pitch_type' column exists, print unique pitch type count and value counts
        print(f"{label}: pitch_types={df['pitch_type'].nunique():,}")
        print(df["pitch_type"].value_counts().head(12))
    return df.head() # Return the first few rows of the DataFrame

# Helper function for merging DataFrames with diagnostic output
def merge_diag(left, right, keys, label):
    print(f"[merge:{label}] before left_rows={len(left):,}, right_rows={len(right):,}, keys={keys}") # Print info before merge
    print(f"[merge:{label}] right_duplicate_keys={right.duplicated(keys).sum():,}") # Check for duplicate keys in the right DataFrame
    # Perform a left merge, validating 'many_to_one' and indicating merge status
    merged = left.merge(right, on=keys, how="left", validate="many_to_one", indicator=True)
    # Print info after merge, including unmatched rows
    print(f"[merge:{label}] after rows={len(merged):,}, unmatched={merged['_merge'].eq('left_only').sum():,}")
    return merged.drop(columns=["_merge"]) # Return the merged DataFrame, dropping the merge indicator column

# Helper function to display content of a CSV file
def show_csv(path, rows=8):
    import pandas as pd # Import pandas locally within the function
    path = BASE_DIR / path # Construct the full path to the CSV file
    print(path)
    df = pd.read_csv(path) # Read the CSV into a pandas DataFrame
    display(df.head(rows)) # Display the first 'rows' of the DataFrame
    return df


In [ ]:
# Run the pitch sequence extension script with the specified TOP_N parameter
run_step([sys.executable, "code/pitch_sequence_extension.py", "--top-n", str(TOP_N)])
# Run the pitch choice diagnostics script with the specified TOP_N parameter
run_step([sys.executable, "code/pitch_choice_diagnostics.py", "--top-n", str(TOP_N)])
temporal = show_csv("output/temporal_validation_2025_to_2026.csv")
sweep = show_csv("output/next_pitch_model_sweep_metrics.csv")
accuracy = show_csv("output/prediction_accuracy_by_pitcher.csv")


Outputs include temporal validation metrics, prediction audits, calibration files, and model-sweep figures.
The per-pitcher state-machine/XGBoost comparison remains the primary model family; this notebook supplies robustness checks.